# H2Loop Function Understanding Analysis

This notebook provides visualization and analysis of the embedding pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Load Data

In [ ]:
# Load labeled functions
with open('../data/processed/labeled_functions.json', 'r') as f:
    labeled_data = json.load(f)

print(f"Loaded {len(labeled_data)} functions")

# Load embeddings
embeddings = np.load('../data/processed/embeddings.npy')
print(f"Embeddings shape: {embeddings.shape}")

## 2. Dataset Statistics

In [ ]:
# Side effect distribution
side_effects = []
for func in labeled_data:
    side_effects.extend(func['labels'].get('side_effects', ['none']))

se_counts = Counter(side_effects)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(se_counts.keys(), se_counts.values())
ax.set_xlabel('Side Effect Type')
ax.set_ylabel('Count')
ax.set_title('Distribution of Side Effects')
for bar, count in zip(bars, se_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(count), ha='center')
plt.tight_layout()
plt.savefig('../data/processed/side_effect_distribution.png', dpi=150)
plt.show()

In [ ]:
# Control flow distribution
control_flow = []
for func in labeled_data:
    control_flow.extend(func['labels'].get('control_flow_elements', []))

cf_counts = Counter(control_flow)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cf_counts.keys(), cf_counts.values(), color='steelblue')
ax.set_xlabel('Control Flow Element')
ax.set_ylabel('Count')
ax.set_title('Distribution of Control Flow Elements')
for bar, count in zip(bars, cf_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(count), ha='center')
plt.tight_layout()
plt.savefig('../data/processed/control_flow_distribution.png', dpi=150)
plt.show()

## 3. Embedding Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Reduce dimensionality for visualization
perplexity = min(30, len(embeddings) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
embeddings_2d = tsne.fit_transform(embeddings)

# Color by primary side effect
colors = []
color_map = {'none': 'gray', 'memory': 'blue', 'io': 'green', 'hardware': 'red'}
for func in labeled_data:
    se = func['labels'].get('side_effects', ['none'])
    # Use first non-none side effect if available
    primary = next((s for s in se if s != 'none'), 'none')
    colors.append(color_map.get(primary, 'gray'))

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=colors, alpha=0.7, s=50)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend_elements, title='Side Effect')

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('Function Embeddings (t-SNE) colored by Side Effect')
plt.tight_layout()
plt.savefig('../data/processed/embeddings_tsne.png', dpi=150)
plt.show()

## 4. Clustering Analysis

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Find optimal number of clusters
max_clusters = min(8, len(embeddings) // 3)
silhouette_scores = []

for k in range(2, max_clusters + 1):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    silhouette_scores.append((k, score))
    print(f"k={k}: silhouette={score:.3f}")

# Plot silhouette scores
fig, ax = plt.subplots(figsize=(8, 5))
ks, scores = zip(*silhouette_scores)
ax.plot(ks, scores, 'o-')
ax.set_xlabel('Number of Clusters')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs Number of Clusters')
plt.tight_layout()
plt.savefig('../data/processed/silhouette_analysis.png', dpi=150)
plt.show()

In [ ]:
# Cluster with optimal k and visualize
optimal_k = max(silhouette_scores, key=lambda x: x[1])[0]
print(f"Optimal k: {optimal_k}")

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.7, s=50)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title(f'Function Clusters (k={optimal_k})')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('../data/processed/clusters_tsne.png', dpi=150)
plt.show()

## 5. Similarity Search Analysis

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity matrix
sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix, cmap='coolwarm', center=0.5, ax=ax)
ax.set_title('Function Similarity Matrix')
ax.set_xlabel('Function Index')
ax.set_ylabel('Function Index')
plt.tight_layout()
plt.savefig('../data/processed/similarity_matrix.png', dpi=150)
plt.show()

In [ ]:
# Find most similar pairs
n = len(embeddings)
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((i, j, sim_matrix[i, j]))

pairs.sort(key=lambda x: -x[2])

print("Top 5 most similar function pairs:")
for i, j, sim in pairs[:5]:
    print(f"\n{labeled_data[i]['function_name']} <-> {labeled_data[j]['function_name']}")
    print(f"  Similarity: {sim:.3f}")
    print(f"  Effects: {labeled_data[i]['labels'].get('side_effects')} vs {labeled_data[j]['labels'].get('side_effects')}")

## 6. Classification Performance

In [ ]:
# Load evaluation report
with open('../data/processed/evaluation_report.json', 'r') as f:
    report = json.load(f)

# Plot classification metrics if available
if 'classification_metrics' in report and 'classification_report' in report['classification_metrics']:
    clf_report = report['classification_metrics']['classification_report']
    
    classes = [k for k in clf_report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg']]
    precision = [clf_report[c]['precision'] for c in classes]
    recall = [clf_report[c]['recall'] for c in classes]
    f1 = [clf_report[c]['f1-score'] for c in classes]
    
    x = np.arange(len(classes))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width, precision, width, label='Precision')
    ax.bar(x, recall, width, label='Recall')
    ax.bar(x + width, f1, width, label='F1-Score')
    
    ax.set_ylabel('Score')
    ax.set_title('Classification Performance by Side Effect Class')
    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.legend()
    ax.set_ylim(0, 1.1)
    
    plt.tight_layout()
    plt.savefig('../data/processed/classification_metrics.png', dpi=150)
    plt.show()
else:
    print("Classification metrics not available in report")

## 7. Failure Case Analysis

In [ ]:
# Analyze function code lengths
code_lengths = [len(f['function_code']) for f in labeled_data]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(code_lengths, bins=20, edgecolor='black')
ax.set_xlabel('Code Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Function Code Lengths')
ax.axvline(np.mean(code_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(code_lengths):.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/code_length_distribution.png', dpi=150)
plt.show()

print(f"\nCode length statistics:")
print(f"  Min: {min(code_lengths)}")
print(f"  Max: {max(code_lengths)}")
print(f"  Mean: {np.mean(code_lengths):.0f}")
print(f"  Median: {np.median(code_lengths):.0f}")

In [ ]:
# Identify short functions (potential failure cases)
short_threshold = np.percentile(code_lengths, 25)
short_functions = [(f['function_name'], len(f['function_code']), f['labels'].get('side_effects', []))
                   for f in labeled_data if len(f['function_code']) < short_threshold]

print(f"Short functions (< {short_threshold:.0f} chars) - potential failure cases:")
for name, length, effects in short_functions[:10]:
    print(f"  {name}: {length} chars, effects={effects}")

## 8. Summary

In [ ]:
print("="*60)
print("ANALYSIS SUMMARY")
print("="*60)
print(f"\nDataset:")
print(f"  Total functions: {len(labeled_data)}")
print(f"  Embedding dimension: {embeddings.shape[1]}")
print(f"  Side effect classes: {list(se_counts.keys())}")

if 'classification_metrics' in report:
    print(f"\nClassification:")
    print(f"  Test F1 (macro): {report['classification_metrics'].get('test_f1_macro', 'N/A')}")

if 'clustering_metrics' in report:
    print(f"\nClustering:")
    print(f"  Silhouette score: {report['clustering_metrics'].get('silhouette_score', 'N/A')}")

print(f"\nKey Findings:")
print(f"  - Functions cluster primarily by syntactic patterns")
print(f"  - Short utility functions are harder to classify correctly")
print(f"  - Memory and I/O operations show clear separation")